# DIY Copilot
## Lesson: Augmentation

A public LLM knows a lot about Python and FastAPI – but nothing about *your* company: your business rules, your engineering guidelines, your security requirements. When you ask it about them, it either admits that it doesn't know – or, much more often, it confidently invents an answer. And code generated on the basis of invented rules violates your real ones.

Coding assistants solve this by putting relevant internal knowledge into the prompt. This is called **augmentation**.

During this lesson you will learn ...

- how to augment prompts with internal knowledge
- about the disadvantages of putting a whole knowledge base into the prompt

### Set up the environment
Import the necessary libraries, set constants, and define helper functions.

In [1]:
# >>> COLAB SETUP <<<
# Placeholder: clone the workshop repository into the Colab workspace and make `lib` importable.
# When running locally, the following line is sufficient.
import sys
sys.path.append('../../../')

In [2]:
from litellm import completion

from lib.completion import generate_completion
from lib.constants import KNOWLEDGE_BASE_PATH
from lib.environment import load_environment
from lib.rag import list_knowledge_base_files

# Load the API key and the default models. If you want, you can change the model.
default_models = load_environment()
MODEL_ID = default_models["generation_model"]
print(f"Using model: {MODEL_ID}")

TEMPERATURE = 0.7
MAX_OUTPUT_TOKENS = 600
SYSTEM_PROMPT = "You are a helpful coding assistant for the developers of our company."

Using model: gemini/gemini-3.1-flash-lite


### Our internal knowledge base
Our (fictional) company has an internal wiki. The documents relevant for developers are:
* **domain**: business rules written by the business departments, e.g. the meeting room booking policy of facility management
* **engineering**: company-wide guidelines for Python code, API design, testing and code reviews
* **security**: binding requirements for secrets, logging, data protection, access and the use of AI assistants

Like every real wiki, it also contains documents for other purposes (**hr**, **it**), similar rules for other locations and resources (Munich office, parking, event spaces) and outdated versions in `archive/` folders.

Let's have a look at the documents.

In [3]:
knowledge_base_files = list_knowledge_base_files(KNOWLEDGE_BASE_PATH)
for file_name in knowledge_base_files:
    print(file_name)

domain/desk-sharing-policy.md
domain/glossary.md
domain/meeting-rooms-berlin.md
domain/room-booking-policy.md
engineering/api-design-guidelines.md
engineering/git-and-code-review.md
engineering/python-coding-guidelines.md
engineering/testing-guidelines.md
security/ai-assistant-usage-policy.md
security/data-protection.md
security/secrets-and-logging.md


In [4]:
# Let's test the model's knowledge about our company guidelines
user_prompt = (
    "I'm building a new internal API for our company. Which HTTP header do I have to use for "
    "the API key, and what must the JSON body of an error response look like?"
)
print(generate_completion(user_prompt=user_prompt, system_prompt=SYSTEM_PROMPT, model_name=MODEL_ID, max_output_tokens=MAX_OUTPUT_TOKENS))

[retry] temporary error (ServiceUnavailableError), waiting 2s (attempt 1/5) ...
To maintain consistency across our internal services, we follow standardized patterns for authentication and error handling. Please implement your API according to the following guidelines:

### 1. API Key Header
Use the standard `X-API-Key` header for authentication. 

*   **Header Name:** `X-API-Key`
*   **Example:** 
    `X-API-Key: your_secret_token_here`

*Note: If your API is strictly internal and requires higher security, consider using `Authorization: Bearer <token>` if you eventually migrate to JWTs or OAuth2, but for simple API key authentication, `X-API-Key` is the company standard.*

---

### 2. Standardized Error Response Body
To ensure our front-end and consuming services can parse errors consistently, all error responses must use a JSON body with the following structure.

**Required fields:**
*   `error`: A machine-readable code (e.g., `INVALID_CREDENTIALS`, `RESOURCE_NOT_FOUND`).
*   `messag

The answer probably sounds plausible – it may suggest `Authorization: Bearer ...` or a common error format like RFC 7807. Now compare it with sections API-04 and API-05 in `engineering/api-design-guidelines.md`: our company uses a custom header `X-<Service>-Key` (because the corporate proxy strips the `Authorization` header!) and its own error format. The model had no chance to know this.

### Exercise 01: Provide internal knowledge by including the whole knowledge base
The simplest way to provide internal knowledge is to include *all* documents in the prompt. Your task is to read all documents of the knowledge base and pass their content alongside the question. Do you find a way of getting insights into the token usage?

**Hints**:
* Read every document and include its content in the user message. Put the file name in front of each document, so the model can refer to it.
* Use the `usage` attribute of the LiteLLM response to inspect token counts

In [6]:
# TODO: Read all documents of the knowledge base, each prefixed with its file name
knowledge_base_content = "\n\n".join(
    f"### Document: {file_name}\n{(KNOWLEDGE_BASE_PATH / file_name).read_text(encoding='utf-8')}"
    for file_name in knowledge_base_files
)
print(f"The knowledge base has {len(knowledge_base_content):,} characters.")

# TODO: Build the prompt with the full knowledge base
full_prompt = f"Based on the following documents:\n\n{knowledge_base_content}\n\nQuestion: {user_prompt}"

response_full = completion(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": full_prompt},
    ],
    temperature=TEMPERATURE,
    max_completion_tokens=MAX_OUTPUT_TOKENS,
)

# TODO: Print insights of the response's token usage
print(f"Input tokens count: {response_full.usage.prompt_tokens}")
print(f"Total tokens count: {response_full.usage.total_tokens}")
print(f"\nAnswer: {response_full.choices[0].message.content}")

The knowledge base has 25,933 characters.
Input tokens count: 6757
Total tokens count: 7042

Answer: Based on the **API Design Guidelines** (`engineering/api-design-guidelines.md`), here are the requirements for your internal API:

### API Key Header
You must use a custom header named **`X-<Service>-Key`**. 
*   *Example:* If your service is named "Roomie", the header would be `X-Roomie-Key`.
*   *Reasoning:* Per **API-04**, the internal corporate reverse proxy strips the standard `Authorization` header, and not all internal clients support SSO.

### Error Response Body
The JSON body of an error response must follow this structure:

```json
{
  "error": {
    "code": "<SERVICE>-<HTTP status>-<REASON>",
    "message": "A human-readable English text."
  }
}
```

**Key requirements for this response:**
*   **`code`**: Must be a stable, machine-readable string. The service prefix (e.g., `ROOM`, `PAY`) must match the one defined in the company's service catalogue.
*   **`message`**: A human

### Exercise 02: Enhance the prompt with only the relevant knowledge
To answer this question, the model only needs two short sections of one document. Your task is to enhance the prompt below by providing only these relevant sections.

In [7]:
# The relevant sections of the API design guidelines, which should be enough to answer the question
relevant_sections = '''
### Document: engineering/api-design-guidelines.md

## API-04 – Authentication
Internal services authenticate clients with API keys. The key is sent in a
custom header named `X-<Service>-Key`, e.g. `X-Roomie-Key`.
Why not `Authorization: Bearer`? The corporate reverse proxy in the internal
network zone strips the `Authorization` header.

## API-05 – Error responses
All errors use the same JSON body:
{"error": {"code": "ROOM-409-OVERLAP", "message": "Room A1-02 is already booked from 09:00 to 10:00."}}
`code` is a stable, machine-readable code of the form `<SERVICE>-<HTTP status>-<REASON>`.
'''

# TODO: Create an augmented prompt where the user prompt is augmented with the relevant sections
augmented_user_prompt = (
    f"Answer the question below based on the context.\n\nContext:\n{relevant_sections}\n\nQuestion: {user_prompt}"
)

response_sections = completion(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": augmented_user_prompt},
    ],
    temperature=TEMPERATURE,
    max_completion_tokens=MAX_OUTPUT_TOKENS,
)

# TODO: Print insights of the response's token usage
print(f"Input tokens count: {response_sections.usage.prompt_tokens}")
print(f"Total tokens count: {response_sections.usage.total_tokens}")
print(f"\nAnswer: {response_sections.choices[0].message.content}")

Input tokens count: 245
Total tokens count: 372

Answer: Based on the provided API design guidelines:

*   **HTTP Header:** You must use a custom header named `X-<Service>-Key` (e.g., `X-Roomie-Key`).
*   **Error Response JSON Body:** It must follow this structure:
    ```json
    {
      "error": {
        "code": "<SERVICE>-<HTTP status>-<REASON>",
        "message": "<A descriptive message>"
      }
    }
    ```
    The `code` field must be a stable, machine-readable string following the format specified above.


### Exercise 03: Calculate costs for both calls
Cost is one factor which should prevent us from putting the whole knowledge base into every prompt. But how much is it exactly? And how much more is it than providing only the relevant sections? Use only input tokens for the calculation and assume the paid tier.

**Hints**:
* Look up the current price per million input tokens for your model on the [pricing page](https://ai.google.dev/gemini-api/docs/pricing).
* A coding assistant sends a request for *every* completion, chat message and agent step – easily thousands of requests per developer and day.

In [8]:
# TODO: Calculate the costs of both calls
def calculate_costs(token_count: int, price_per_million_tokens: float) -> float:
    """Calculates the costs of a given number of tokens based on the price per million tokens."""
    return token_count * (price_per_million_tokens / 1_000_000)


# TODO: Set the current price in USD per million input tokens of your model (see pricing page)
usd_per_million_tokens = 0.10  # example value - check the pricing page!

input_tokens_full = response_full.usage.prompt_tokens
input_tokens_sections = response_sections.usage.prompt_tokens

costs_full = calculate_costs(input_tokens_full, usd_per_million_tokens)
costs_sections = calculate_costs(input_tokens_sections, usd_per_million_tokens)

print(f"Costs for {input_tokens_full} tokens using the whole knowledge base: {costs_full:0.6f} USD")
print(f"Costs for {input_tokens_sections} tokens using the relevant sections: {costs_sections:0.6f} USD")
print(f"Using the whole knowledge base is {round(costs_full / costs_sections)} times more expensive.")

Costs for 6757 tokens using the whole knowledge base: 0.000676 USD
Costs for 245 tokens using the relevant sections: 0.000025 USD
Using the whole knowledge base is 28 times more expensive.


In [9]:
# TODO: Extrapolate - what would 1,000 requests per developer and day cost for a team of 10 developers?
requests_per_day = 1_000 * 10
print(f"Whole knowledge base: {costs_full * requests_per_day:,.2f} USD per day")
print(f"Relevant sections:    {costs_sections * requests_per_day:,.2f} USD per day")

Whole knowledge base: 6.76 USD per day
Relevant sections:    0.25 USD per day


### Learnings
* Without internal knowledge, the model hallucinates plausible but wrong answers about your company – and generates code that violates your guidelines.
* Our knowledge base is tiny (about 20 documents). A real company wiki has thousands of pages – it would not even fit into the context window of the model.
* Even if it fits, more context means higher **costs**, higher **latency**, and the relevant information can get lost in the noise.
* So the key question is: **how do we find the few relevant sections automatically?** That's what *retrieval* is about – see the next notebook.